# Generación de Modelos de IA con LangChain

LangChain es un framework de código abierto diseñado para construir aplicaciones impulsadas por modelos de lenguaje (LLMs). Su principal fortaleza reside en proporcionar una **interfaz unificada** para interactuar con múltiples proveedores de IA, facilitando el encadenamiento de operaciones, la gestión de memoria, la integración con herramientas externas y la construcción de agentes inteligentes.

---

## ¿Qué es LangChain?

LangChain actúa como una capa de abstracción sobre los diferentes proveedores de LLMs. Sus características principales son:

- **Interfaz unificada**: cambia de OpenAI a Anthropic o Gemini con mínimos cambios de código.
- **Chains**: encadena múltiples operaciones de forma declarativa.
- **Agentes**: modelos que deciden dinámicamente qué herramientas usar.
- **Memory**: gestión de historial de conversaciones.
- **RAG**: integración nativa con bases de datos vectoriales para recuperación de contexto.
- **LCEL** (LangChain Expression Language): sintaxis de composición fluida con operador `|`.

---

## Instalación

### Paquete base

```bash
pip install langchain
```

### Paquetes por proveedor

```bash
# OpenAI
pip install langchain-openai

# Anthropic
pip install langchain-anthropic

# Google Gemini
pip install langchain-google-genai

# Ollama (modelos locales)
pip install langchain-ollama

# Herramientas adicionales comunes
pip install langchain-community python-dotenv
```

### Gestión de claves API con dotenv
```{index} dotenv
```
La librería python-dotenv sirve para leer datos, por ejemplo claves apis que se encuentran en un fichero denominado.env

```bash
# Archivo .env
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
GOOGLE_API_KEY=AIza...
```

```python
from dotenv import load_dotenv
load_dotenv()
```

---

## Conceptos Fundamentales

### La interfaz ChatModel

Todos los modelos en LangChain comparten la misma interfaz base. Esto significa que el resto de tu código (chains, agentes, memoria) funciona igual independientemente del proveedor.

```python
# Todos estos objetos son intercambiables
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

# Todos exponen los mismos métodos: .invoke(), .stream(), .batch()
```

### Tipos de mensajes

```python
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="Eres un experto en ciencia de datos."),
    HumanMessage(content="¿Qué es el sobreajuste?"),
]
```

### PromptTemplates

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente especializado en {materia}."),
    ("human", "{pregunta}"),
])

# Rellenar variables
messages = prompt.format_messages(
    materia="física cuántica",
    pregunta="¿Qué es la superposición?"
)
```

### Output Parsers

```python
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

# Parser de texto plano
parser = StrOutputParser()

# Parser de JSON estructurado
json_parser = JsonOutputParser()
```

---

## OpenAI con LangChain

### Configuración básica

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7,
    max_tokens=1024,
    # api_key se lee automáticamente de OPENAI_API_KEY
)

response = llm.invoke([
    SystemMessage(content="Responde siempre en español."),
    HumanMessage(content="¿Cuál es la diferencia entre ML y DL?")
])

print(response.content)
```

### Modelos disponibles de OpenAI

| Modelo | Identificador | Características |
|---|---|---|
| GPT-4o | `gpt-4o` | Multimodal, más capaz |
| GPT-4o Mini | `gpt-4o-mini` | Rápido y económico |
| GPT-4 Turbo | `gpt-4-turbo` | Contexto 128K tokens |
| o1 | `o1` | Razonamiento avanzado |
| o3 Mini | `o3-mini` | Razonamiento eficiente |

### Streaming de respuestas

```python
llm = ChatOpenAI(model="gpt-4o", streaming=True)

for chunk in llm.stream("Escribe un poema sobre la IA."):
    print(chunk.content, end="", flush=True)
```

### Chain básica con LCEL

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor profesional."),
    ("human", "Traduce al {idioma}: {texto}"),
])

chain = prompt | llm | StrOutputParser()

resultado = chain.invoke({
    "idioma": "francés",
    "texto": "La inteligencia artificial cambia el mundo."
})

print(resultado)
```

### Salida estructurada con OpenAI

```python
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class Resumen(BaseModel):
    titulo: str = Field(description="Título del resumen")
    puntos_clave: list[str] = Field(description="Lista de puntos clave")
    conclusion: str = Field(description="Conclusión final")

llm = ChatOpenAI(model="gpt-4o")
llm_estructurado = llm.with_structured_output(Resumen)

resumen = llm_estructurado.invoke(
    "Resume los beneficios del aprendizaje automático en medicina."
)

print(resumen.titulo)
print(resumen.puntos_clave)
```

### Visión (imágenes) con GPT-4o

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o")

mensaje = HumanMessage(content=[
    {"type": "text", "text": "¿Qué aparece en esta imagen?"},
    {
        "type": "image_url",
        "image_url": {"url": "https://ejemplo.com/imagen.jpg"}
    }
])

response = llm.invoke([mensaje])
print(response.content)
```

---

## Ollama con LangChain

Ollama permite ejecutar modelos de forma completamente local, garantizando privacidad total y sin costes por API. Para ver la parte teórica de estos modelos, [ir al apartado de este enlace](ollama2).

### Configuración básica

```python
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0.5,
    # Por defecto conecta a http://localhost:11434
)

response = llm.invoke("¿Qué es LangChain?")
print(response.content)
```

### Modelos recomendados para Ollama

| Modelo | Comando pull | Especialidad |
|---|---|---|
| Llama 3.1 8B | `ollama pull llama3.1` | Propósito general |
| Llama 3.1 70B | `ollama pull llama3.1:70b` | Alta calidad |
| Mistral 7B | `ollama pull mistral` | Rápido y eficiente |
| DeepSeek-R1 | `ollama pull deepseek-r1` | Razonamiento |
| Phi-4 | `ollama pull phi4` | Hardware limitado |
| Gemma 3 | `ollama pull gemma3` | Google open-source |
| LLaVA | `ollama pull llava` | Visión multimodal |

### Chain con modelo local

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.1", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en {dominio}. Responde de forma concisa."),
    ("human", "{pregunta}"),
])

chain = prompt | llm | StrOutputParser()

respuesta = chain.invoke({
    "dominio": "desarrollo web",
    "pregunta": "¿Cuáles son las ventajas de usar TypeScript?"
})

print(respuesta)
```

### Streaming local

```python
from langchain_ollama import ChatOllama

llm = ChatOllama(model="mistral")

for chunk in llm.stream("Explica la arquitectura transformer paso a paso."):
    print(chunk.content, end="", flush=True)
```

### Embeddings locales con Ollama

```python
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Generar embedding de un texto
vector = embeddings.embed_query("Aprendizaje profundo con redes neuronales")
print(f"Dimensiones: {len(vector)}")  # 768

# Embeddings de múltiples documentos
vectores = embeddings.embed_documents([
    "LangChain facilita el desarrollo con LLMs.",
    "Ollama ejecuta modelos localmente.",
    "Los transformers revolucionaron el NLP.",
])
```

### Cambio dinámico de modelo (fallback)

```python
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI

# Intenta con el modelo local; si falla, usa OpenAI
llm_local = ChatOllama(model="llama3.1")
llm_nube = ChatOpenAI(model="gpt-4o-mini")

llm_con_fallback = llm_local.with_fallbacks([llm_nube])

response = llm_con_fallback.invoke("¿Qué es RAG en IA?")
print(response.content)
```

---

## Google Gemini con LangChain

### Configuración básica

```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
    # api_key se lee de GOOGLE_API_KEY
)

response = llm.invoke("¿Cómo funciona la atención en los transformers?")
print(response.content)
```

### Modelos disponibles de Gemini

| Modelo | Identificador | Características |
|---|---|---|
| Gemini 2.0 Flash | `gemini-2.0-flash` | Rápido, multimodal |
| Gemini 2.5 Pro | `gemini-2.5-pro-preview-05-06` | Máxima capacidad |
| Gemini 1.5 Flash | `gemini-1.5-flash` | Contexto 1M tokens |
| Gemini 1.5 Pro | `gemini-1.5-pro` | Análisis profundo |

### Chain con Gemini

```python
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.5)

prompt = ChatPromptTemplate.from_template(
    "Actúa como un tutor de {materia}. Explica {concepto} con un ejemplo práctico."
)

chain = prompt | llm | StrOutputParser()

respuesta = chain.invoke({
    "materia": "estadística",
    "concepto": "la desviación estándar"
})
print(respuesta)
```

### Visión multimodal con Gemini

```python
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import base64

# Leer imagen local
with open("diagrama.png", "rb") as f:
    imagen_b64 = base64.b64encode(f.read()).decode()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

mensaje = HumanMessage(content=[
    {
        "type": "image_url",
        "image_url": {"url": f"data:image/png;base64,{imagen_b64}"}
    },
    {"type": "text", "text": "Describe este diagrama de arquitectura."}
])

response = llm.invoke([mensaje])
print(response.content)
```

### Embeddings de Google

```python
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vector = embeddings.embed_query("Redes neuronales convolucionales")
print(f"Dimensiones: {len(vector)}")  # 768

# Para documentos
vectores = embeddings.embed_documents([
    "Primer documento sobre IA.",
    "Segundo documento sobre ML.",
])
```

### Salida estructurada con Gemini

```python
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

class AnalisisSentimiento(BaseModel):
    sentimiento: str = Field(description="positivo, negativo o neutro")
    puntuacion: float = Field(description="Puntuación entre -1.0 y 1.0")
    razon: str = Field(description="Explicación breve del análisis")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
analizador = llm.with_structured_output(AnalisisSentimiento)

resultado = analizador.invoke(
    "Me encanta cómo LangChain simplifica el trabajo con LLMs."
)

print(f"Sentimiento: {resultado.sentimiento}")
print(f"Puntuación: {resultado.puntuacion}")
```

---

## Anthropic Claude con LangChain

### Configuración básica

```python
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0.7,
    max_tokens=2048,
    # api_key se lee de ANTHROPIC_API_KEY
)

response = llm.invoke("Explica el concepto de embeddings vectoriales.")
print(response.content)
```

### Modelos disponibles de Anthropic

| Modelo | Identificador | Características |
|---|---|---|
| Claude Sonnet 4.5 | `claude-sonnet-4-5` | Equilibrio calidad/velocidad |
| Claude Opus 4.5 | `claude-opus-4-5` | Máxima capacidad |
| Claude Haiku 3.5 | `claude-haiku-3-5` | Ultrarrápido y económico |

### Chain con Claude

```python
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-sonnet-4-5", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un revisor de código experto. 
    Analiza el código, identifica problemas y sugiere mejoras claras."""),
    ("human", "Revisa este código:\n\n```{lenguaje}\n{codigo}\n```"),
])

chain = prompt | llm | StrOutputParser()

revision = chain.invoke({
    "lenguaje": "python",
    "codigo": """
def calcular_promedio(numeros):
    suma = 0
    for n in numeros:
        suma = suma + n
    return suma / len(numeros)
    """
})

print(revision)
```

### Conversación con memoria usando Claude

```python
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatAnthropic(model="claude-sonnet-4-5")

# Historial de conversación (gestionado manualmente)
historial = [
    SystemMessage(content="Eres un asistente de programación en Python.")
]

def chat(pregunta: str) -> str:
    historial.append(HumanMessage(content=pregunta))
    respuesta = llm.invoke(historial)
    historial.append(AIMessage(content=respuesta.content))
    return respuesta.content

print(chat("¿Cómo se usa un decorador en Python?"))
print(chat("¿Puedes darme un ejemplo más avanzado?"))
print(chat("¿Cómo lo aplicaría para medir el tiempo de ejecución?"))
```

### Salida estructurada con Claude

```python
from pydantic import BaseModel, Field
from langchain_anthropic import ChatAnthropic
from typing import List

class PlanAprendizaje(BaseModel):
    tema: str = Field(description="Tema principal del plan")
    duracion_semanas: int = Field(description="Duración estimada en semanas")
    objetivos: List[str] = Field(description="Objetivos de aprendizaje")
    recursos: List[str] = Field(description="Recursos recomendados")
    proyectos: List[str] = Field(description="Proyectos prácticos sugeridos")

llm = ChatAnthropic(model="claude-sonnet-4-5")
planificador = llm.with_structured_output(PlanAprendizaje)

plan = planificador.invoke(
    "Crea un plan de aprendizaje para dominar LangChain desde cero."
)

print(f"Tema: {plan.tema}")
print(f"Duración: {plan.duracion_semanas} semanas")
print("Objetivos:", plan.objetivos)
```

---

## Chains Avanzadas con LCEL

LCEL (LangChain Expression Language) usa el operador `|` para componer componentes de forma declarativa.

### Chain secuencial

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

# Primer chain: genera una pregunta
prompt_pregunta = ChatPromptTemplate.from_template(
    "Formula una pregunta difícil sobre {tema}."
)

# Segundo chain: responde la pregunta
prompt_respuesta = ChatPromptTemplate.from_template(
    "Responde esta pregunta de forma experta: {pregunta}"
)

parser = StrOutputParser()

# Chain anidada
chain_completa = (
    prompt_pregunta
    | llm
    | parser
    | (lambda pregunta: {"pregunta": pregunta})
    | prompt_respuesta
    | llm
    | parser
)

resultado = chain_completa.invoke({"tema": "redes neuronales recurrentes"})
print(resultado)
```

### Chain paralela con RunnableParallel

```python
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# Ejecutar múltiples chains en paralelo
analisis_paralelo = RunnableParallel(
    ventajas=(
        ChatPromptTemplate.from_template("Lista las ventajas de {tecnologia}.") 
        | llm | parser
    ),
    desventajas=(
        ChatPromptTemplate.from_template("Lista las desventajas de {tecnologia}.")
        | llm | parser
    ),
    casos_de_uso=(
        ChatPromptTemplate.from_template("Describe casos de uso de {tecnologia}.")
        | llm | parser
    ),
)

resultado = analisis_paralelo.invoke({"tecnologia": "LangChain"})

print("VENTAJAS:", resultado["ventajas"])
print("DESVENTAJAS:", resultado["desventajas"])
print("CASOS DE USO:", resultado["casos_de_uso"])
```

---

## RAG (Retrieval-Augmented Generation)
```{index} RAG, Retrieval-Augmented
```

RAG combina la recuperación de documentos relevantes con la generación de respuestas, permitiendo que el modelo acceda a conocimiento externo y actualizado.

### RAG básico con FAISS

```bash
pip install faiss-cpu langchain-community
```

```python
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Documentos de ejemplo
documentos_texto = [
    "LangChain es un framework para construir aplicaciones con LLMs.",
    "LCEL permite encadenar componentes usando el operador pipe |.",
    "Los agentes en LangChain pueden usar herramientas externas dinámicamente.",
    "RAG mejora las respuestas al recuperar contexto relevante de documentos.",
    "LangSmith permite depurar y monitorizar aplicaciones LangChain.",
]

# Dividir en chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(documentos_texto)

# Crear vectorstore
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Prompt RAG
prompt = ChatPromptTemplate.from_template("""
Responde la pregunta basándote ÚNICAMENTE en el siguiente contexto:

{contexto}

Pregunta: {pregunta}
""")

llm = ChatOpenAI(model="gpt-4o-mini")

# Chain RAG completa
chain_rag = (
    {"contexto": retriever, "pregunta": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

respuesta = chain_rag.invoke("¿Qué es LCEL en LangChain?")
print(respuesta)
```

### RAG con Ollama (completamente local y gratuito)

```python
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Embeddings y LLM 100% locales
embeddings = OllamaEmbeddings(model="nomic-embed-text")
llm = ChatOllama(model="llama3.1", temperature=0.1)

# Crear vectorstore local
textos = ["Documento uno...", "Documento dos...", "Documento tres..."]
vectorstore = FAISS.from_texts(textos, embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_template("""
Usa el siguiente contexto para responder.
Contexto: {contexto}
Pregunta: {pregunta}
""")

chain = (
    {"contexto": retriever, "pregunta": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(chain.invoke("¿Qué información hay en los documentos?"))
```

---

## Agentes con LangChain

Los agentes permiten al modelo decidir dinámicamente qué herramientas usar para resolver una tarea.

### Agente con herramientas personalizadas

```python
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import datetime

@tool
def obtener_fecha_actual() -> str:
    """Devuelve la fecha y hora actual."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática. Ejemplo: '2 + 2 * 10'"""
    try:
        resultado = eval(expresion)
        return f"El resultado de {expresion} es {resultado}"
    except Exception as e:
        return f"Error al calcular: {str(e)}"

@tool
def buscar_definicion(termino: str) -> str:
    """Proporciona una definición básica de un término de IA."""
    definiciones = {
        "llm": "Modelo de Lenguaje Grande: modelo entrenado con grandes cantidades de texto.",
        "rag": "Retrieval-Augmented Generation: técnica que combina recuperación y generación.",
        "embedding": "Representación vectorial densa de texto en un espacio de alta dimensión.",
    }
    return definiciones.get(termino.lower(), f"No tengo definición para '{termino}'.")

# Configurar el agente
llm = ChatOpenAI(model="gpt-4o", temperature=0)
herramientas = [obtener_fecha_actual, calcular, buscar_definicion]

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil con acceso a herramientas."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agente = create_tool_calling_agent(llm, herramientas, prompt)
ejecutor = AgentExecutor(agent=agente, tools=herramientas, verbose=True)

resultado = ejecutor.invoke({
    "input": "¿Qué fecha es hoy? ¿Y cuánto es 15 al cuadrado?"
})
print(resultado["output"])
```

---

## Memoria y Gestión del Historial

### Memoria con RunnableWithMessageHistory

```python
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

llm = ChatAnthropic(model="claude-sonnet-4-5")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente conversacional con memoria."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm

# Almacén de sesiones en memoria
store: dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_con_memoria = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "usuario_001"}}

# El historial se mantiene automáticamente entre llamadas
r1 = chain_con_memoria.invoke({"input": "Me llamo Ana."}, config=config)
r2 = chain_con_memoria.invoke({"input": "¿Cómo me llamo?"}, config=config)

print(r1.content)
print(r2.content)  # Recordará el nombre "Ana"
```

---

## Comparativa de Proveedores en LangChain

| Característica | OpenAI | Ollama | Google Gemini | Anthropic |
|---|---|---|---|---|
| Paquete | `langchain-openai` | `langchain-ollama` | `langchain-google-genai` | `langchain-anthropic` |
| Clase principal | `ChatOpenAI` | `ChatOllama` | `ChatGoogleGenerativeAI` | `ChatAnthropic` |
| Ejecución | Nube | Local | Nube | Nube |
| Coste | Por tokens | Gratuito | Por tokens (free tier) | Por tokens |
| Privacidad | Externa | Total (local) | Externa | Externa |
| Embeddings | `OpenAIEmbeddings` | `OllamaEmbeddings` | `GoogleGenerativeAIEmbeddings` | — |
| Multimodal | ✅ GPT-4o | ✅ LLaVA | ✅ Gemini | ✅ Claude 3+ |
| Contexto máx. | 128K tokens | Según modelo | 1M tokens | 200K tokens |

---

## Depuración con LangSmith

LangSmith es la plataforma oficial de observabilidad para aplicaciones LangChain.

```bash
pip install langsmith
```

```bash
# Variables de entorno
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__...
LANGCHAIN_PROJECT=mi-proyecto-ia
```

```python
# Con estas variables configuradas, todas las ejecuciones
# se registran automáticamente en LangSmith sin cambios de código.

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke("¿Qué es LangSmith?")
# Esta llamada queda registrada en el dashboard de LangSmith
```

---

## Buenas Prácticas

- **Usa `with_structured_output`** para extraer datos estructurados en lugar de parsear texto manualmente; es más robusto y tipado.
- **Prefiere LCEL** sobre las chains heredadas (`LLMChain`); es más flexible, soporta streaming y es el estándar actual.
- **Ollama para desarrollo y pruebas**: usa modelos locales durante el desarrollo para reducir costes y proteger datos sensibles.
- **Gestiona el contexto**: los LLMs tienen límites de tokens; usa `RecursiveCharacterTextSplitter` con solapamiento para no perder coherencia entre chunks.
- **Temperatura según tarea**: análisis y extracción → `0.0–0.2`; respuestas equilibradas → `0.5–0.7`; escritura creativa → `0.8–1.0`.
- **Fallbacks**: define cadenas de respaldo con `.with_fallbacks()` para mayor resiliencia en producción.
- **Cachea embeddings**: en pipelines RAG, almacena los vectores en disco (FAISS, Chroma) en lugar de regenerarlos en cada ejecución.
- **Monitoriza con LangSmith**: es indispensable en producción para detectar latencias, tokens consumidos y errores en las chains.

---

## Recursos Adicionales

- Documentación oficial: [https://python.langchain.com](https://python.langchain.com)
- LangChain en GitHub: [https://github.com/langchain-ai/langchain](https://github.com/langchain-ai/langchain)
- LangSmith: [https://smith.langchain.com](https://smith.langchain.com)
- Hub de prompts: [https://smith.langchain.com/hub](https://smith.langchain.com/hub)
- API OpenAI: [https://platform.openai.com/docs](https://platform.openai.com/docs)
- API Anthropic: [https://docs.anthropic.com](https://docs.anthropic.com)
- API Google Gemini: [https://ai.google.dev/docs](https://ai.google.dev/docs)
- Modelos Ollama: [https://ollama.com/library](https://ollama.com/library)